# PIXEL AI -- retrain VAE + GAN on Kaggle GPU

Runs the project's actual training scripts (`vae/train.py`, `gan/train.py`) against a GPU instead of CPU, with higher epoch budgets, then regenerates the exact result images used in the project README by re-executing the same notebooks the repo already ships.

**Before running:** in the panel on the right, go to **Settings -> Accelerator** and pick **GPU P100**. Our training scripts only use a single GPU (no multi-GPU code), so P100 is the faster single-GPU option here -- "GPU T4 x2" would leave the second T4 idle. If P100 isn't available (quota/availability), T4 x2 still works fine, just a bit slower.

Then **Run All**. Everything below is self-contained -- it clones the public GitHub repo, trains, and zips the results for download. Nothing here needs the original machine.

## 1. Check the GPU we actually got

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No GPU detected -- go to Settings (right panel) -> Accelerator -> GPU P100, "
    "then Run All again."
)
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA available:", torch.cuda.is_available())


## 2. Clone the repo

Public repo, no credentials needed.

In [ ]:
!git clone https://github.com/Udbhav748/PIXEL-AI.git
%cd PIXEL-AI


## 3. Install anything missing

Kaggle's base image already has a GPU-enabled `torch`/`torchvision` and `matplotlib`/`numpy`/`pillow` -- deliberately *not* reinstalling those so we don't accidentally overwrite Kaggle's CUDA-matched build with a CPU-only one from pip. `streamlit` and `pyoidn` aren't needed for training (OIDN has nothing to train), so they're skipped here too.

In [ ]:
import torch, torchvision
print("torch:", torch.__version__, "| torchvision:", torchvision.__version__)


## 4. Config -- edit these if you want

GPU training is roughly 10-20x faster than the CPU runs this project was originally trained with, so these epoch counts are pushed well beyond what was practical locally. The GAN learning rates use **TTUR** (`--lr-d` lower than `--lr`) -- the discriminator update slower than the generator, which our local CPU experiments showed tends to overpower the generator otherwise (see `gan/README.md`).

In [ ]:
VAE_EPOCHS = 50
VAE_LATENT_DIM = 32  # up from the CPU-trained default of 20 -- more capacity to
                     # preserve digit identity through the encode/decode round-trip

GAN_EPOCHS = 150
GAN_SAMPLE_EVERY = 10
GAN_LR_G = 2e-4
GAN_LR_D = 1e-4  # TTUR: discriminator updates slower than the generator


## 5. Train the VAE

Note: the checkpoint already in the repo used `latent_dim=20`; this run uses 32 (see config above), so it's a genuinely different, higher-capacity model, not just "the same one, longer."

In [ ]:
!python vae/train.py --epochs {VAE_EPOCHS} --latent-dim {VAE_LATENT_DIM}


## 6. Train the GAN -- MLP (basic)

Fresh weights each run (DCGAN-paper-style init, N(0, 0.02) -- see `weights_init()` in `gan/train.py`), not resumed from the CPU-trained checkpoint already in the repo, so this is a clean, properly-tuned run rather than continuing from a differently-configured one.

In [ ]:
!python gan/train.py --architecture mlp --epochs {GAN_EPOCHS} --lr {GAN_LR_G} --lr-d {GAN_LR_D} --sample-every {GAN_SAMPLE_EVERY}


## 7. Train the GAN -- DCGAN (conv)

In [ ]:
!python gan/train.py --architecture dcgan --epochs {GAN_EPOCHS} --lr {GAN_LR_G} --lr-d {GAN_LR_D} --sample-every {GAN_SAMPLE_EVERY}


## 8. Regenerate the exact result images used in the README

Re-runs the project's own notebooks against the freshly trained checkpoints -- same process used locally, so the output files land in exactly the same place (`results/vae/*.png`, `results/gan/*.png`) with no separate scripts to maintain.

In [ ]:
!jupyter nbconvert --to notebook --execute --inplace notebooks/02_VAE.ipynb
!jupyter nbconvert --to notebook --execute --inplace notebooks/03_GAN.ipynb


## 9. Zip everything to bring back to your machine

Includes the new checkpoints, the regenerated result images, and the re-executed notebooks (so the printed metrics/AUC/accuracy numbers travel with them).

In [ ]:
!zip -r /kaggle/working/pixel_ai_retrained.zip models results notebooks/*.ipynb
print("Done -- download pixel_ai_retrained.zip from the Output tab on the right.")


## 10. Bringing it back into your local repo

1. Download `pixel_ai_retrained.zip` from this notebook's **Output** tab.
2. Unzip it.
3. Copy its `models/`, `results/`, and `notebooks/*.ipynb` into your local `PIXEL-AI/` folder, overwriting the existing ones.
4. From your local repo: `git add -A && git commit -m "Retrain VAE/GAN on Kaggle GPU with more epochs" && git push`.

The root `README.md`'s numbers (PSNR, accuracy, AUC, class coverage) were written against the CPU-trained checkpoints, so after this you'll want to re-check them against whatever the re-executed notebooks print now and update the text if the numbers moved.